# 02 — Benchmark hỏi–đáp có kiểm chứng cho MỘT phiên bản ATT&CK

**Đề tài:** Trôi phiên bản tri thức trong trợ lý CTI dùng LLM/RAG
**Notebook này chạy HAI lần**: một lần với `ATTACK_VERSION = "19.2"` (chuẩn hiện hành), một lần với `"18.1"` (chuẩn cũ). Mỗi lần ghi kết quả vào thư mục riêng `benchmark/attack-<phiên bản>/`, nên hai lần chạy không ghi đè lên nhau.

| Notebook | Việc | Đầu ra |
|---|---|---|
| `01_thay_doi_phien_ban_ATTACK` | Ground truth các thay đổi giữa các phiên bản ATT&CK | `drift/` |
| **`02_benchmark_theo_phien_ban` (notebook này)** | Bộ câu hỏi–đáp án có kiểm chứng cho một phiên bản | `benchmark/attack-<phiên bản>/` |
| `03` (tiếp theo) | Ghép cặp câu hỏi trước/sau theo kiểu thay đổi | bộ câu hỏi cho Trụ 2 |
| `04` | Pilot: LLM trả lời không RAG / RAG một phiên bản / RAG trộn phiên bản | tỉ lệ mã lỗi thời theo kiểu thay đổi |
| `05` | Phương pháp nhận biết phiên bản (Trụ 3) | so sánh với các baseline |

## Benchmark ở đây nghĩa là gì?

Một **benchmark** là bộ dữ liệu đánh giá **cố định, có phiên bản, ai chạy lại cũng ra đúng kết quả**. Mỗi mục gồm:

1. **Câu hỏi** mà nhà phân tích SOC có thể hỏi trợ lý CTI, ví dụ: *"T1055.011 thuộc tactic nào?"*.
2. **Đáp án đúng (gold) theo đúng phiên bản đang xét**, lấy **tất định** từ dữ liệu có cấu trúc (quan hệ STIX, bản ghi CVE), **không do LLM viết**.
3. **Tài liệu bằng chứng (evidence)**: các đoạn trong kho RAG chứa sự kiện đó, dùng để đo khả năng truy hồi.
4. **Đáp án nhiễu (distractor)**: một đáp án **sai nhưng cùng kiểu** (một tactic khác, một CWE khác…), dùng để phân tích *kiểu* sai của mô hình.

Vì đáp án đúng **phụ thuộc phiên bản**, cùng một câu hỏi có thể có đáp án khác nhau ở v18.1 và v19.2. Chênh lệch đó chính là dữ liệu của đề tài.

## Vì sao làm như vậy là đúng chuẩn?

- **Sinh câu hỏi theo mẫu từ đồ thị tri thức** là phương pháp chuẩn của cộng đồng KGQA (ví dụ LC-QuAD, MetaQA): đáp án đúng *theo cấu tạo*.
- **Định dạng đầu ra tương thích BEIR** (`corpus.jsonl`, `queries.jsonl`, `qrels/*.tsv`), định dạng chuẩn để đánh giá truy hồi.
- **Tái lập**: phiên bản ghim, mỗi file nguồn có SHA-256, seed cố định, có `manifest.json` và `DATASHEET.md`.
- **Chia tập theo thực thể** để tránh rò rỉ giữa dev và test.
- **Chấm điểm bằng ID chuẩn** (T1055, M1040, CWE-502…), không dùng LLM làm giám khảo.

### Nhật ký phiên bản

| Phiên bản | Ngày | Thay đổi |
|---|---|---|
| 1.0 | 12/10/2026 | Bản đầu tiên |
| 1.1 | 12/10/2026 | Q5 chỉ giữ CVE mà CWE là *Weakness* hợp lệ (loại các mã *category* cũ như CWE-189, CWE-264, CWE-399); bắt buộc có danh mục CWE; ghi phiên bản CWE vào manifest. Mỗi mẫu câu hỏi dùng bộ sinh ngẫu nhiên riêng, nên sửa một mẫu không làm thay đổi các mẫu khác. Vì đổi cách sinh ngẫu nhiên, toàn bộ câu hỏi khác bản 1.0. |
| 1.2 | 12/10/2026 | Chuyển sang đề tài trôi phiên bản: kết quả ghi theo thư mục `benchmark/attack-<phiên bản>/`; dùng chung thư mục ATT&CK với notebook 01; đổi `target` thành `distractor`; bài kiểm thử chấm điểm tự nhận biết phiên bản. Nội dung câu hỏi và đáp án không đổi so với 1.1. |
| 1.3 | 13/10/2026 | Chạy được với ATT&CK v6.3 (chưa có sub-technique): ô xem thử đoạn T1055.011 không còn báo lỗi khi mã này không tồn tại. Câu hỏi và đáp án của mọi phiên bản không đổi. |

## 1. Cấu hình

Chạy notebook **trong thư mục dự án** (ví dụ `~/p/cti-drift/`). Mọi đường dẫn đều tương đối với thư mục đó.

- `ATTACK_VERSION`: phiên bản ATT&CK của lần chạy này. Chạy lần lượt với `"19.2"` rồi `"18.1"`. Mỗi phiên bản có thư mục kết quả riêng; dữ liệu nguồn (KEV, CVE, CAPEC, CWE) dùng chung nên giống hệt nhau giữa hai lần chạy.
- `MAX_KEV_CVES = None`: dùng toàn bộ danh mục KEV. Đặt một số nhỏ (ví dụ 100) khi chỉ muốn chạy thử nhanh.
- `SEED`: mọi phép chọn ngẫu nhiên đều đi qua seed này.

In [1]:
from pathlib import Path
import json, re, hashlib, random, zipfile, io, datetime, collections, textwrap
import concurrent.futures as cf
import requests
import pandas as pd
import networkx as nx

ATTACK_VERSION    = "7.2"     # ĐỔI Ở ĐÂY: "19.2", "18.1"; thí nghiệm khẳng định (notebook 07) cần thêm "6.3" và "7.2"

ROOT       = Path.cwd()
RAW        = ROOT / "data" / "raw"                                   # dữ liệu nguồn dùng chung
PROC       = ROOT / "data" / "processed" / f"attack-{ATTACK_VERSION}"  # đồ thị tri thức theo phiên bản
BENCH      = ROOT / "benchmark" / f"attack-{ATTACK_VERSION}"          # benchmark theo phiên bản
for p in (RAW, RAW / "attack_versions", PROC, BENCH, BENCH / "qrels"):
    p.mkdir(parents=True, exist_ok=True)

SEED              = 2026
MAX_KEV_CVES      = None       # None = toàn bộ KEV
MAX_PER_TEMPLATE  = 400        # trần số câu hỏi mỗi mẫu (trước khi chia tập)
DEV_PERCENT       = 20         # % thực thể đưa vào tập dev
CHUNK_MAX_WORDS   = 300        # độ dài tối đa một đoạn (chunk) trong kho RAG
HUMAN_AUDIT_N     = 200        # số câu đưa đi kiểm tra thủ công
FETCH_WORKERS     = 16
REQUIRE_CWE_CATALOG = True     # bản chính thức BẮT BUỘC có danh mục CWE (v1.1)
BENCHMARK_VERSION = "1.3"

pd.set_option("display.max_colwidth", 90)
print("ROOT =", ROOT, "| ATT&CK", ATTACK_VERSION, "| ghi vào", BENCH.relative_to(ROOT))

ROOT = ~/p/cti-drift | ATT&CK 7.2 | ghi vào benchmark/attack-7.2


## 2. Tải và đóng băng dữ liệu nguồn

| Nguồn | Vai trò trong benchmark | Nơi lấy |
|---|---|---|
| MITRE ATT&CK Enterprise (STIX 2.1) | Kỹ thuật, tactic, biện pháp giảm thiểu (mitigation), nhóm tấn công | `mitre-attack/attack-stix-data`, file đã ghim phiên bản |
| MITRE CAPEC (STIX 2.1) | Cầu nối CWE ↔ CAPEC ↔ ATT&CK | `mitre/cti` |
| CISA KEV | Danh sách CVE đang bị khai thác thực tế, kèm CWE | bản sao chính thức `cisagov/kev-data` |
| CVE List V5 | Mô tả CVE, CVSS (của CNA hoặc CISA-ADP Vulnrichment) | `CVEProject/cvelistV5` |
| CWE (tuỳ chọn) | Tên và mô tả điểm yếu | `cwe.mitre.org` (nếu không tải được, lấy tên CWE từ bản ghi CVE) |

**Vì sao chọn tập con KEV thay vì toàn bộ NVD (hơn 300.000 CVE)?** KEV gồm các CVE *đang bị khai thác*, đúng loại nhà phân tích SOC hay hỏi, và đủ nhỏ (khoảng 1.700) để kiểm soát chất lượng. Lý do này phải ghi trong bài báo.

Mọi file được lưu vào `data/raw/` cùng mã băm SHA-256. Lần chạy sau dùng lại file đã lưu (cache), nên kết quả không đổi kể cả khi nguồn cập nhật.

In [2]:
MANIFEST = {"benchmark_version": BENCHMARK_VERSION, "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
            "seed": SEED, "attack_version": ATTACK_VERSION, "sources": {}}

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

def fetch(url: str, dest: Path, timeout=180) -> Path:
    # Tải một lần; lần sau dùng bản đã lưu để bảo đảm tái lập
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        dest.write_bytes(r.content)
    return dest

def register(name, url, path, **extra):
    MANIFEST["sources"][name] = {"url": url, "file": str(path.relative_to(ROOT)),
                                 "sha256": sha256_file(path), **extra}

URLS = {
    "attack": f"https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/enterprise-attack/enterprise-attack-{ATTACK_VERSION}.json",
    "capec":  "https://raw.githubusercontent.com/mitre/cti/master/capec/2.1/stix-capec.json",
    "kev":    "https://raw.githubusercontent.com/cisagov/kev-data/main/known_exploited_vulnerabilities.json",
    "cwe":    "https://cwe.mitre.org/data/xml/cwec_latest.xml.zip",
}
p_attack = fetch(URLS["attack"], RAW / "attack_versions" / f"enterprise-attack-{ATTACK_VERSION}.json"); register("attack", URLS["attack"], p_attack)
p_capec  = fetch(URLS["capec"],  RAW / "stix-capec.json");                          register("capec",  URLS["capec"],  p_capec)
p_kev    = fetch(URLS["kev"],    RAW / "known_exploited_vulnerabilities.json")
kev_json = json.loads(p_kev.read_text())
register("kev", URLS["kev"], p_kev, catalog_version=kev_json["catalogVersion"], count=kev_json["count"])

try:
    p_cwe = fetch(URLS["cwe"], RAW / "cwec_latest.xml.zip", timeout=60)
    register("cwe", URLS["cwe"], p_cwe)
except Exception as ex:
    p_cwe = None
    if REQUIRE_CWE_CATALOG:
        raise RuntimeError("Không tải được danh mục CWE — bản chính thức cần file này để lọc CWE hợp lệ") from ex
    print("⚠ Không tải được danh mục CWE (chế độ thử nghiệm):", type(ex).__name__)

for k, v in MANIFEST["sources"].items():
    print(f"{k:7s} sha256={v['sha256'][:16]}…  {v.get('catalog_version', '')}")

attack  sha256=04728e4b995ee7e2…  
capec   sha256=ee6244f48259c196…  
kev     sha256=fb1621beec7db19d…  2026.10.08
cwe     sha256=3976f599e5e52002…  


### 2.1 Tải bản ghi CVE cho các mục KEV

Mỗi CVE trong KEV được lấy từ CVE List V5 (định dạng JSON 5.x). Từ đó trích **mô tả** và **điểm CVSS v3.x**, ưu tiên điểm do CNA (bên cấp mã CVE) công bố, nếu không có thì lấy điểm do CISA-ADP (Vulnrichment) bổ sung. **Nguồn của điểm được ghi vào từng bản ghi** để phản biện kiểm tra được.

In [3]:
kev = kev_json["vulnerabilities"]
if MAX_KEV_CVES:
    kev = sorted(kev, key=lambda v: v["cveID"])[:MAX_KEV_CVES]

def cvelist_url(cve_id):
    _, year, num = cve_id.split("-")
    return f"https://raw.githubusercontent.com/CVEProject/cvelistV5/main/cves/{year}/{num[:-3]}xxx/{cve_id}.json"

def fetch_cve(cve_id):
    dest = RAW / "cvelistV5" / f"{cve_id}.json"
    try:
        fetch(cvelist_url(cve_id), dest, timeout=60)
        return cve_id, dest, None
    except Exception as ex:
        return cve_id, None, f"{type(ex).__name__}: {ex}"[:120]

with cf.ThreadPoolExecutor(FETCH_WORKERS) as ex:
    fetched = list(ex.map(fetch_cve, [v["cveID"] for v in kev]))
cve_paths = {c: p for c, p, e in fetched if p}
cve_errors = {c: e for c, p, e in fetched if e}
MANIFEST["sources"]["cvelistV5"] = {"url_pattern": cvelist_url("CVE-YYYY-NNNNN"),
                                    "n_records": len(cve_paths), "n_failed": len(cve_errors),
                                    "sha256_of_sorted_hashes": hashlib.sha256("".join(sha256_file(cve_paths[c]) for c in sorted(cve_paths)).encode()).hexdigest()}
print(f"Tải được {len(cve_paths)} / {len(kev)} bản ghi CVE; lỗi: {len(cve_errors)}")
list(cve_errors.items())[:5]

Tải được 1739 / 1739 bản ghi CVE; lỗi: 0


[]

## 3. Phân tích ATT&CK (STIX 2.1)

Quy tắc làm sạch:
- Bỏ các đối tượng **revoked** hoặc **deprecated**. Đây là nguồn nhiễu phổ biến; nhiều bài báo quên bước này.
- Lấy ID chuẩn (T/TA/M/G/S) từ `external_references` với `source_name = "mitre-attack"`.
- Xoá chú thích `(Citation: …)` và rút link markdown về dạng chữ, để văn bản đưa vào RAG sạch.

**Lưu ý quan trọng với ATT&CK v19:** tactic TA0005 nay tên là **Stealth** (trước đây là *Defense Evasion*), và có tactic mới **Defense Impairment** (TA0112). LLM được huấn luyện trên dữ liệu cũ nhiều khả năng sẽ trả lời theo tên cũ. Vì vậy notebook 04 **bắt buộc** phải có baseline closed-book, để tách phần "LLM tự biết" khỏi phần "LLM học từ RAG". Đây cũng là một luận điểm hay cho phần động cơ của bài: trợ lý CTI *phải* dựa vào truy hồi, vì tri thức thay đổi theo phiên bản.

In [4]:
CITATION_RE = re.compile(r"\(Citation:[^)]*\)")
MDLINK_RE   = re.compile(r"\[([^\]]+)\]\((?:https?://[^)]+)\)")
WS_RE       = re.compile(r"[ \t]+")

def clean_text(s: str) -> str:
    s = CITATION_RE.sub("", s or "")
    s = MDLINK_RE.sub(r"\1", s)
    s = s.replace("<code>", "").replace("</code>", "")
    s = WS_RE.sub(" ", s)
    return re.sub(r"\s+([.,;:])", r"\1", s).strip()

def ext_id(o, src="mitre-attack"):
    for r in o.get("external_references", []):
        if r.get("source_name") == src and r.get("external_id"):
            return r["external_id"]
    return None

def is_active(o):
    return not o.get("revoked", False) and not o.get("x_mitre_deprecated", False)

attack_objs = json.loads(p_attack.read_text())["objects"]

tactics = {}            # shortname -> (TA id, name)
for o in attack_objs:
    if o["type"] == "x-mitre-tactic" and is_active(o):
        tactics[o["x_mitre_shortname"]] = (ext_id(o), o["name"])
TACTIC_NAME = {tid: name for tid, name in tactics.values()}

stix2id, techniques, mitigations, groups, software = {}, {}, {}, {}, {}
for o in attack_objs:
    if not is_active(o):
        continue
    t, i = o["type"], ext_id(o)
    if not i:
        continue
    if t == "attack-pattern":
        tac = [tactics[k["phase_name"]][0] for k in o.get("kill_chain_phases", [])
               if k.get("kill_chain_name") == "mitre-attack" and k["phase_name"] in tactics]
        techniques[i] = dict(id=i, name=o["name"], description=clean_text(o.get("description", "")),
                             tactics=sorted(set(tac)), platforms=o.get("x_mitre_platforms", []),
                             is_sub=o.get("x_mitre_is_subtechnique", False), parent=None)
    elif t == "course-of-action":
        mitigations[i] = dict(id=i, name=o["name"], description=clean_text(o.get("description", "")))
    elif t == "intrusion-set":
        groups[i] = dict(id=i, name=o["name"], aliases=[a for a in o.get("aliases", []) if a != o["name"]],
                         description=clean_text(o.get("description", "")))
    elif t in ("malware", "tool"):
        software[i] = dict(id=i, name=o["name"], kind=t)
    else:
        continue
    stix2id[o["id"]] = i

# Quan hệ: chỉ giữ khi cả hai đầu đều đang hoạt động
mitigates, uses_tech, uses_sw = collections.defaultdict(dict), collections.defaultdict(dict), collections.defaultdict(set)
for o in attack_objs:
    if o["type"] != "relationship" or not is_active(o):
        continue
    s, d, rt = stix2id.get(o["source_ref"]), stix2id.get(o["target_ref"]), o["relationship_type"]
    if not s or not d:
        continue
    desc = clean_text(o.get("description", ""))
    if rt == "subtechnique-of" and s in techniques and d in techniques:
        techniques[s]["parent"] = d
    elif rt == "mitigates" and s in mitigations and d in techniques:
        mitigates[s][d] = desc
    elif rt == "uses" and s in groups and d in techniques:
        uses_tech[s][d] = desc
    elif rt == "uses" and s in groups and d in software:
        uses_sw[s].add(d)

tech_mits = collections.defaultdict(set)
for m, ts in mitigates.items():
    for t in ts:
        tech_mits[t].add(m)

print(f"Tactic: {len(TACTIC_NAME)} | Kỹ thuật: {len(techniques)} (sub: {sum(t['is_sub'] for t in techniques.values())})"
      f" | Mitigation: {len(mitigations)} | Nhóm: {len(groups)} | Phần mềm: {len(software)}")
print(f"Cạnh mitigates: {sum(map(len, mitigates.values()))} | group→technique: {sum(map(len, uses_tech.values()))}")
pd.Series(TACTIC_NAME).to_frame("tactic")

Tactic: 12 | Kỹ thuật: 428 (sub: 272) | Mitigation: 41 | Nhóm: 107 | Phần mềm: 410
Cạnh mitigates: 798 | group→technique: 1722


,tactic
TA0009,Collection
TA0011,Command and Control
TA0006,Credential Access
TA0005,Defense Evasion
TA0007,Discovery
TA0002,Execution
TA0010,Exfiltration
TA0040,Impact
TA0001,Initial Access
TA0008,Lateral Movement


## 4. Phân tích CAPEC, KEV, CVE và CWE

CAPEC là **cầu nối giữa hai thế giới**: phía lỗ hổng (CVE → CWE) và phía hành vi tấn công (ATT&CK). Đường đi `CVE → CWE → CAPEC → ATT&CK` là thứ phương pháp nhận biết phiên bản (notebook 05) có thể khai thác: một tài liệu nói *"CVE-X là lỗi SQL injection và được khai thác qua T1003"* có thể bị đối chiếu xem đường nối đó có tồn tại trong đồ thị hay không.

**Nguồn sự thật cho từng trường** (ghi vào datasheet):
- CWE của CVE: trường `cwes` trong KEV, do CISA biên tập. Bỏ các giá trị không phải `CWE-<số>`.
- CVSS v3.x: bản ghi CVE, ưu tiên CNA, rồi đến CISA-ADP. Không dùng điểm CVSS v2 hay v4 để thang đo nhất quán.

In [5]:
# ---- CAPEC
capec = {}
for o in json.loads(p_capec.read_text())["objects"]:
    if o["type"] != "attack-pattern" or o.get("revoked") or o.get("x_capec_status") in ("Deprecated", "Obsolete"):
        continue
    cid = ext_id(o, "capec")
    if not cid:
        continue
    refs = o.get("external_references", [])
    capec[cid] = dict(id=cid, name=o["name"], description=clean_text(o.get("description", "")),
                      cwes=sorted({r["external_id"] for r in refs if r.get("source_name") == "cwe"}),
                      techniques=sorted({r["external_id"] for r in refs
                                         if r.get("source_name") == "ATTACK" and r.get("external_id") in techniques}))

# ---- CVE (KEV + CVE List V5)
CWE_RE = re.compile(r"^CWE-\d+$")

def pick_cvss(rec):
    # Ưu tiên CNA, rồi CISA-ADP; chỉ lấy CVSS v3.1/v3.0
    conts = [("CNA", rec["containers"].get("cna", {}))] + \
            [(a.get("providerMetadata", {}).get("shortName", "ADP"), a) for a in rec["containers"].get("adp", [])]
    for src, cont in conts:
        for m in cont.get("metrics", []) or []:
            for key in ("cvssV3_1", "cvssV3_0"):
                if key in m and "baseScore" in m[key]:
                    return dict(score=float(m[key]["baseScore"]), severity=m[key]["baseSeverity"].upper(),
                                vector=m[key].get("vectorString", ""), source=src)
    return None

def cna_description(rec):
    for d in rec["containers"].get("cna", {}).get("descriptions", []):
        if d.get("lang", "").startswith("en"):
            return clean_text(d["value"])
    return ""

cves, cwe_names_from_cve = {}, {}
for v in kev:
    cid = v["cveID"]
    rec = json.loads(cve_paths[cid].read_text()) if cid in cve_paths else None
    if rec:
        for pt in rec["containers"].get("cna", {}).get("problemTypes", []):
            for d in pt.get("descriptions", []):
                if CWE_RE.match(d.get("cweId", "") or "") and d.get("description"):
                    cwe_names_from_cve.setdefault(d["cweId"], re.sub(r"^CWE-\d+[:\s]*", "", d["description"]).strip())
    cves[cid] = dict(id=cid, vendor=v["vendorProject"].strip(), product=v["product"].strip(),
                     title=v["vulnerabilityName"].strip(), kev_desc=clean_text(v["shortDescription"]),
                     date_added=v["dateAdded"], ransomware=v["knownRansomwareCampaignUse"],
                     cwes=sorted({c for c in v.get("cwes", []) if CWE_RE.match(c)}),
                     description=cna_description(rec) if rec else "",
                     cvss=pick_cvss(rec) if rec else None)

# ---- CWE: danh mục chính thức nếu có, không thì lấy tên từ bản ghi CVE
cwe = {}
if p_cwe:
    try:
        import xml.etree.ElementTree as ET
        with zipfile.ZipFile(p_cwe) as z:
            root = ET.fromstring(z.read([n for n in z.namelist() if n.endswith(".xml")][0]))
        strip = lambda tag: tag.split("}")[-1]
        for w in root.iter():
            if strip(w.tag) != "Weakness" or w.get("Status") == "Deprecated":
                continue
            desc = next((c.text or "" for c in w if strip(c.tag) == "Description"), "")
            cwe[f"CWE-{w.get('ID')}"] = dict(id=f"CWE-{w.get('ID')}", name=w.get("Name"),
                                             description=clean_text(desc), is_weakness=True)
        MANIFEST["sources"]["cwe"]["n_weaknesses"] = len(cwe)
        MANIFEST["sources"]["cwe"]["catalog_version"] = root.get("Version")
    except Exception as ex:
        print("⚠ Lỗi đọc CWE XML, dùng tên từ CVE:", ex)
        cwe = {}
used_cwes = {c for v in cves.values() for c in v["cwes"]} | {c for p in capec.values() for c in p["cwes"]}
for c in sorted(used_cwes):
    if c not in cwe:
        cwe[c] = dict(id=c, name=cwe_names_from_cve.get(c, ""), description="", is_weakness=False)
CWE_CATALOG_OK = any(c.get("is_weakness") for c in cwe.values())
n_nonweak = sum(1 for v in cves.values() for c in v["cwes"] if not cwe[c]["is_weakness"])
print(f"Liên kết CVE→CWE trỏ tới mục KHÔNG phải Weakness (category/lỗi thời): {n_nonweak}" if CWE_CATALOG_OK else "")

n_cvss = sum(v["cvss"] is not None for v in cves.values())
print(f"CAPEC: {len(capec)} (có ánh xạ ATT&CK: {sum(bool(p['techniques']) for p in capec.values())})")
print(f"CVE (KEV): {len(cves)} | có CWE: {sum(bool(v['cwes']) for v in cves.values())} | có CVSS v3: {n_cvss}")
print("Nguồn CVSS:", collections.Counter(v["cvss"]["source"] for v in cves.values() if v["cvss"]))
print(f"CWE: {len(cwe)} (có tên: {sum(bool(c['name']) for c in cwe.values())})")

Liên kết CVE→CWE trỏ tới mục KHÔNG phải Weakness (category/lỗi thời): 63
CAPEC: 558 (có ánh xạ ATT&CK: 164)
CVE (KEV): 1739 | có CWE: 1564 | có CVSS v3: 1667
Nguồn CVSS: Counter({'CISA-ADP': 990, 'CNA': 675, 'redhat-SADP': 2})
CWE: 952 (có tên: 944)


## 5. Dựng đồ thị tri thức (Knowledge Graph — KG)

KG là **sổ đăng ký tri thức của đúng phiên bản này**. Phương pháp nhận biết phiên bản (notebook 05) sẽ đối chiếu câu trả lời với KG của phiên bản hiện hành. Mỗi nút có `type`, mỗi cạnh có `rel`.

| Cạnh | Ý nghĩa |
|---|---|
| technique —`in_tactic`→ tactic | kỹ thuật thuộc tactic |
| technique —`subtechnique_of`→ technique | quan hệ cha–con |
| mitigation —`mitigates`→ technique | biện pháp giảm thiểu |
| group —`uses`→ technique / software | nhóm tấn công dùng kỹ thuật |
| cve —`has_weakness`→ cwe | CVE thuộc loại điểm yếu |
| capec —`targets_weakness`→ cwe | mẫu tấn công khai thác điểm yếu |
| capec —`maps_to`→ technique | ánh xạ CAPEC sang ATT&CK |

**Câu phản biện chắc chắn gặp:** *"Nếu đã có KG chứa đáp án thì cần RAG làm gì?"* Cách trả lời:
1. Trợ lý trả lời câu hỏi ngôn ngữ tự nhiên trên kho văn bản lẫn lộn nhiều phiên bản (báo cáo, advisory, blog); KG chỉ phủ phần sự kiện đã chuẩn hoá.
2. KG được dùng làm **bộ kiểm tra theo phiên bản**, không làm bộ trả lời.
3. Ở notebook 05 sẽ có thí nghiệm **giảm độ phủ KG** để đo phương pháp còn hiệu quả đến đâu khi KG không đầy đủ.

In [6]:
KG = nx.MultiDiGraph()
for tid, name in TACTIC_NAME.items():
    KG.add_node(tid, type="tactic", name=name)
for t in techniques.values():
    KG.add_node(t["id"], type="technique", name=t["name"], is_sub=t["is_sub"])
for m in mitigations.values():
    KG.add_node(m["id"], type="mitigation", name=m["name"])
for g in groups.values():
    KG.add_node(g["id"], type="group", name=g["name"])
for s in software.values():
    KG.add_node(s["id"], type="software", name=s["name"])
for c in cwe.values():
    KG.add_node(c["id"], type="cwe", name=c["name"])
for p in capec.values():
    KG.add_node(p["id"], type="capec", name=p["name"])
for v in cves.values():
    attrs = dict(type="cve", name=v["title"], vendor=v["vendor"], product=v["product"])
    if v["cvss"]:
        attrs.update(cvss_score=v["cvss"]["score"], cvss_severity=v["cvss"]["severity"], cvss_source=v["cvss"]["source"])
    KG.add_node(v["id"], **attrs)

for t in techniques.values():
    for ta in t["tactics"]:
        KG.add_edge(t["id"], ta, rel="in_tactic")
    if t["parent"]:
        KG.add_edge(t["id"], t["parent"], rel="subtechnique_of")
for m, ts in mitigates.items():
    for t in ts:
        KG.add_edge(m, t, rel="mitigates")
for g, ts in uses_tech.items():
    for t in ts:
        KG.add_edge(g, t, rel="uses")
for g, ss in uses_sw.items():
    for s in ss:
        KG.add_edge(g, s, rel="uses")
for v in cves.values():
    for c in v["cwes"]:
        KG.add_edge(v["id"], c, rel="has_weakness")
for p in capec.values():
    for c in p["cwes"]:
        if c in KG:
            KG.add_edge(p["id"], c, rel="targets_weakness")
    for t in p["techniques"]:
        KG.add_edge(p["id"], t, rel="maps_to")

def kg_targets(node, rel):
    return sorted({d for _, d, r in KG.out_edges(node, data="rel") if r == rel})

print(KG)
print("Loại nút:", collections.Counter(nx.get_node_attributes(KG, "type").values()))
print("Loại cạnh:", collections.Counter(r for *_, r in KG.edges(data="rel")))

# Độ phủ đường đi xuyên phân loại CVE → CWE → CAPEC → ATT&CK (số liệu dùng được cho bài báo)
cwe2capec = collections.defaultdict(set)
for p in capec.values():
    for c in p["cwes"]:
        cwe2capec[c].add(p["id"])
reach = sum(any(capec[pc]["techniques"] for c in v["cwes"] for pc in cwe2capec.get(c, ())) for v in cves.values())
print(f"CVE trong KEV nối được tới ít nhất một kỹ thuật ATT&CK qua CWE→CAPEC: {reach}/{len(cves)} ({reach/len(cves):.1%})")

MultiDiGraph with 4247 nodes and 7020 edges
Loại nút: Counter({'cve': 1739, 'cwe': 952, 'capec': 558, 'technique': 428, 'software': 410, 'group': 107, 'mitigation': 41, 'tactic': 12})
Loại cạnh: Counter({'uses': 2256, 'has_weakness': 1679, 'targets_weakness': 1211, 'mitigates': 798, 'in_tactic': 575, 'subtechnique_of': 272, 'maps_to': 229})
CVE trong KEV nối được tới ít nhất một kỹ thuật ATT&CK qua CWE→CAPEC: 441/1739 (25.4%)


In [7]:
# Lưu KG: danh sách nút/cạnh dạng CSV (dễ đọc) + node-link JSON (nạp lại bằng networkx)
KG_DIR = PROC / "kg"; KG_DIR.mkdir(exist_ok=True)
pd.DataFrame([{"id": n, **a} for n, a in KG.nodes(data=True)]).to_csv(KG_DIR / "nodes.csv", index=False)
pd.DataFrame([{"src": s, "rel": r, "dst": d} for s, d, r in KG.edges(data="rel")]).to_csv(KG_DIR / "edges.csv", index=False)
try:
    nl = nx.node_link_data(KG, edges="links")      # networkx ≥ 3.4
except TypeError:
    nl = nx.node_link_data(KG)                     # networkx cũ hơn
(KG_DIR / "kg_node_link.json").write_text(json.dumps(nl))
print("Đã lưu KG vào", KG_DIR.relative_to(ROOT))

Đã lưu KG vào data/processed/attack-7.2/kg


## 6. Dựng kho văn bản sạch cho RAG và sổ ghi bằng chứng

Mỗi thực thể được "kết xuất" (render) thành một tài liệu tiếng Anh, theo cách trang web ATT&CK, CAPEC, KEV trình bày, rồi được cắt thành các đoạn (chunk) dài tối đa `CHUNK_MAX_WORDS` từ.

**Điểm kỹ thuật then chốt: sổ ghi bằng chứng (`fact → chunk`).** Mỗi dòng văn bản nêu một sự kiện của KG đều mang theo khoá của sự kiện đó, ví dụ `("T1055.011", "in_tactic", "TA0005")`. Khi cắt đoạn, sổ ghi lại sự kiện nằm ở đoạn nào. Nhờ vậy, mỗi câu hỏi biết **chính xác** đoạn nào chứa bằng chứng, nên recall truy hồi đo được mà không phải gán nhãn tay.

Mỗi đoạn đều lặp lại **dòng tiêu đề** của thực thể (ví dụ `ATT&CK Technique T1055.011: Extra Window Memory Injection`) để đoạn nào cũng tự đứng được, giống cách các hệ RAG thực tế gắn metadata.

In [8]:
corpus, fact2chunks = [], collections.defaultdict(set)

def split_long(text, max_words):
    # Cắt đoạn văn dài theo ranh giới câu
    sents = re.split(r"(?<=[.!?])\s+", text)
    out, cur = [], []
    for s in sents:
        if cur and len(" ".join(cur + [s]).split()) > max_words:
            out.append(" ".join(cur)); cur = []
        cur.append(s)
    if cur:
        out.append(" ".join(cur))
    return out

def emit(doc_id, kind, header, segments):
    # segments: list[(text, [fact_keys])]; đóng gói thành các chunk, ghi sổ bằng chứng
    budget = CHUNK_MAX_WORDS - len(header.split())
    expanded = []
    for text, facts in segments:
        if not text:
            continue
        parts = split_long(text, budget) if len(text.split()) > budget else [text]
        expanded += [(p, facts) for p in parts]
    chunks, cur, cur_facts = [], [], set()
    for text, facts in expanded:
        if cur and len(" ".join(cur + [text]).split()) > budget:
            chunks.append((cur, cur_facts)); cur, cur_facts = [], set()
        cur.append(text); cur_facts |= set(facts)
    if cur:
        chunks.append((cur, cur_facts))
    for k, (texts, facts) in enumerate(chunks):
        cid = f"{doc_id}#{k}"
        corpus.append(dict(_id=cid, doc_id=doc_id, kind=kind, title=header, text=header + "\n" + "\n".join(texts)))
        for f in facts:
            fact2chunks[f].add(cid)

def short(s, n):
    w = s.split()
    return " ".join(w[:n]) + (" …" if len(w) > n else "")

tname = lambda t: f"{t} {techniques[t]['name']}"

for t in techniques.values():
    segs = [(f"Tactics: " + ", ".join(f"{TACTIC_NAME[a]} ({a})" for a in t["tactics"]) + ".",
             [(t["id"], "in_tactic", a) for a in t["tactics"]])]
    if t["parent"]:
        segs.append((f"Sub-technique of: {tname(t['parent'])}.", [(t["id"], "subtechnique_of", t["parent"])]))
    if t["platforms"]:
        segs.append(("Platforms: " + ", ".join(t["platforms"]) + ".", []))
    segs.append((t["description"], []))
    ms = sorted(tech_mits.get(t["id"], ()))
    if ms:
        segs.append(("Mitigations: " + "; ".join(f"{m} {mitigations[m]['name']}" for m in ms) + ".",
                     [(m, "mitigates", t["id"]) for m in ms]))
    emit(t["id"], "technique", f"ATT&CK Technique {t['id']}: {t['name']}", segs)

for m in mitigations.values():
    segs = [(m["description"], [])]
    for t, d in sorted(mitigates.get(m["id"], {}).items()):
        segs.append((f"Mitigates {tname(t)}: {short(d, 50)}", [(m["id"], "mitigates", t)]))
    emit(m["id"], "mitigation", f"ATT&CK Mitigation {m['id']}: {m['name']}", segs)

for g in groups.values():
    segs = []
    if g["aliases"]:
        segs.append(("Associated names: " + ", ".join(g["aliases"]) + ".", []))
    segs.append((g["description"], []))
    for t, d in sorted(uses_tech.get(g["id"], {}).items()):
        segs.append((f"Uses {tname(t)}: {short(d, 40)}", [(g["id"], "uses", t)]))
    emit(g["id"], "group", f"ATT&CK Group {g['id']}: {g['name']}", segs)

for v in cves.values():
    segs = [(f"Vendor/product: {v['vendor']} {v['product']}.", [])]
    if v["cwes"]:
        segs.append(("Weakness: " + "; ".join(f"{c} {cwe[c]['name']}".strip() for c in v["cwes"]) + ".",
                     [(v["id"], "has_weakness", c) for c in v["cwes"]]))
    if v["cvss"]:
        c = v["cvss"]
        segs.append((f"CVSS v3 base score {c['score']:.1f} ({c['severity']}), vector {c['vector']}.",
                     [(v["id"], "cvss_severity", c["severity"])]))
    segs.append((v["description"] or v["kev_desc"], []))
    segs.append((f"Added to the CISA KEV catalog on {v['date_added']}; known ransomware campaign use: {v['ransomware']}.", []))
    emit(v["id"], "cve", f"{v['id']}: {v['title']}", segs)

for c in cwe.values():
    segs = [(c["description"], [])]
    rel = sorted(cwe2capec.get(c["id"], ()))
    if rel:
        segs.append(("Related attack patterns: " + "; ".join(f"{p} {capec[p]['name']}" for p in rel) + ".",
                     [(p, "targets_weakness", c["id"]) for p in rel]))
    emit(c["id"], "cwe", f"{c['id']}: {c['name']}".strip(": "), segs)

for p in capec.values():
    segs = [(p["description"], [])]
    if p["cwes"]:
        segs.append(("Related weaknesses: " + "; ".join(f"{c} {cwe[c]['name']}".strip() for c in p["cwes"]) + ".",
                     [(p["id"], "targets_weakness", c) for c in p["cwes"]]))
    if p["techniques"]:
        segs.append(("Mapped ATT&CK techniques: " + "; ".join(tname(t) for t in p["techniques"]) + ".",
                     [(p["id"], "maps_to", t) for t in p["techniques"]]))
    emit(p["id"], "capec", f"CAPEC {p['id']}: {p['name']}", segs)

doc2chunks = collections.defaultdict(list)
for ch in corpus:
    doc2chunks[ch["doc_id"]].append(ch["_id"])

cdf = pd.DataFrame(corpus)
cdf["n_words"] = cdf.text.str.split().str.len()
print(f"Kho RAG: {len(cdf)} đoạn từ {cdf.doc_id.nunique()} tài liệu; sự kiện có bằng chứng: {len(fact2chunks)}")
cdf.groupby("kind").agg(docs=("doc_id", "nunique"), chunks=("_id", "count"), words_median=("n_words", "median"))

Kho RAG: 4017 đoạn từ 3815 tài liệu; sự kiện có bằng chứng: 8153


,docs,chunks,words_median
kind,,,
capec,556,556,91.5
cve,1739,1754,82.0
cwe,944,944,35.5
group,107,190,256.5
mitigation,41,110,277.0
technique,428,463,141.0


In [9]:
# Xem thử một đoạn
# T1055.011 chỉ có từ v7 trở đi; với bản cũ hơn (ví dụ v6.3) thì xem đoạn đầu tiên
print(next((c for c in corpus if c["_id"] == "T1055.011#0"), corpus[0])["text"][:900])

ATT&CK Technique T1055.011: Extra Window Memory Injection
Tactics: Privilege Escalation (TA0004), Defense Evasion (TA0005).
Sub-technique of: T1055 Process Injection.
Platforms: Windows.
Adversaries may inject malicious code into process via Extra Window Memory (EWM) in order to evade process-based defenses as well as possibly elevate privileges. EWM injection is a method of executing arbitrary code in the address space of a separate live process. Before creating a window, graphical Windows-based processes must prescribe to or register a windows class, which stipulate appearance and behavior (via windows procedures, which are functions that handle input/output of data). Registration of new windows classes can include a request for up to 40 bytes of EWM to be appended to the allocated memory of each instance of that class. This EWM is intended to store data specific to that window and has


## 7. Mẫu câu hỏi và nhóm hậu quả vận hành

Mỗi mẫu (template) gắn với **một quan hệ trong KG** và **một nhóm hậu quả** nếu trợ lý trả lời sai (ví dụ do dùng tri thức lỗi thời):

| Mã | Nhóm hậu quả | Hậu quả vận hành nếu trợ lý trả lời sai | Mẫu |
|---|---|---|---|
| H1 | Quy kết sai kỹ thuật/tactic | Phân tích sai giai đoạn tấn công, sai playbook | Q1, Q2, Q7 |
| H2 | Sai khuyến nghị giảm thiểu | Triển khai biện pháp không chặn được kỹ thuật | Q3 |
| H3 | Sai mức nghiêm trọng | Ưu tiên vá sai: lỗ hổng nguy hiểm bị xếp sau | Q6 |
| H4 | Liên kết sai điểm yếu | Chọn sai biện pháp sửa lỗi gốc | Q5 |
| H5 | Quy kết sai nhóm tấn công | Sai đánh giá đối thủ, sai săn mối đe doạ | Q4 |

**Đáp án nhiễu (distractor)** luôn sai và **cùng kiểu** với đáp án đúng, dùng để phân tích *kiểu* sai của mô hình. Cách chọn được cố định:
- Q2 (cha của sub-technique) và Q7 (ánh xạ CAPEC): distractor là một kỹ thuật **có chung tactic**, tức một đáp án sai nhưng có vẻ hợp lý (hard negative).
- Q6 (mức nghiêm trọng): distractor là mức **xa nhất** trên thang thứ bậc (CRITICAL/HIGH → LOW; MEDIUM/LOW → CRITICAL).
- Các mẫu khác: chọn ngẫu nhiên có seed trong tập giá trị cùng kiểu.

**Q4 dùng giả định thế giới đóng (closed-world).** ATT&CK không liệt kê đầy đủ mọi kỹ thuật một nhóm từng dùng, nên câu hỏi được diễn đạt *"Theo MITRE ATT&CK, nhóm G có được ghi nhận là dùng T không?"*. Khi đó đáp án "không" được định nghĩa rõ là *không có trong ATT&CK*. Các câu "không" dùng **hard negative**: kỹ thuật cùng tactic với một kỹ thuật nhóm đó có dùng.

Mỗi mẫu có 3 cách diễn đạt; mỗi câu hỏi được gán ngẫu nhiên (có seed) một cách, và cả 3 cách đều được lưu để notebook 04 kiểm tra độ nhạy theo cách hỏi.

In [10]:
rng = random.Random(SEED)
SEV_ORDER = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
SEV_TARGET = {"CRITICAL": "LOW", "HIGH": "LOW", "MEDIUM": "CRITICAL", "LOW": "CRITICAL"}
ALL_TACTICS = sorted(TACTIC_NAME)
ALL_MITS = sorted(mitigations)
KEV_CWE_POOL = sorted({c for v in cves.values() for c in v["cwes"] if cwe[c]["is_weakness"] or not CWE_CATALOG_OK})
tech_by_tactic = collections.defaultdict(set)
for t in techniques.values():
    for a in t["tactics"]:
        tech_by_tactic[a].add(t["id"])
parents = sorted(t["id"] for t in techniques.values() if not t["is_sub"])

def same_tactic_distractor(gold_ids, pool=None, r=None):
    # Kỹ thuật sai nhưng có chung tactic với đáp án đúng (hard negative)
    r = r or rng
    cands = set()
    for g in gold_ids:
        for a in techniques[g]["tactics"]:
            cands |= tech_by_tactic[a]
    cands -= set(gold_ids)
    if pool is not None:
        cands &= set(pool)
    return r.choice(sorted(cands)) if cands else None

TEMPLATES = {
  "Q1_tech_tactic": dict(harm="H1", answer_type="tactic_set", phrasings=[
      "Which MITRE ATT&CK tactic(s) does the technique {name} ({id}) belong to?",
      "Under which ATT&CK Enterprise tactic(s) is {id} ({name}) categorized?",
      "In the ATT&CK Enterprise matrix, what tactic(s) is the technique {name} ({id}) listed under?"]),
  "Q2_subtech_parent": dict(harm="H1", answer_type="technique_set", phrasings=[
      "In MITRE ATT&CK, the sub-technique \"{name}\" belongs to which parent technique?",
      "What is the parent ATT&CK technique of the sub-technique named \"{name}\"?",
      "Which ATT&CK technique has \"{name}\" as one of its sub-techniques?"]),
  "Q3_tech_mitigation": dict(harm="H2", answer_type="mitigation_set", phrasings=[
      "Which MITRE ATT&CK mitigations are recommended against {name} ({id})?",
      "What ATT&CK mitigations (M-IDs) does MITRE list for the technique {id} ({name})?",
      "To defend against {name} ({id}), which ATT&CK mitigations apply?"]),
  "Q4_group_uses": dict(harm="H5", answer_type="yes_no", phrasings=[
      "According to MITRE ATT&CK, is the group {gname} ({gid}) documented as using {tname} ({tid})?",
      "Does MITRE ATT&CK record that {gname} ({gid}) has used the technique {tid} ({tname})?",
      "Per ATT&CK, has the threat group {gid} ({gname}) been reported to use {tname} ({tid})? Answer yes or no."]),
  "Q5_cve_cwe": dict(harm="H4", answer_type="cwe_set", phrasings=[
      "Which CWE weakness is associated with {id} ({vendor} {product})?",
      "What is the CWE classification of {id}, the {vendor} {product} vulnerability?",
      "{id} affecting {vendor} {product} maps to which CWE?"]),
  "Q6_cve_severity": dict(harm="H3", answer_type="severity", phrasings=[
      "What is the CVSS v3 base severity of {id} ({vendor} {product})?",
      "How severe is {id} in {vendor} {product} according to its CVSS v3 base score: LOW, MEDIUM, HIGH or CRITICAL?",
      "What CVSS v3 severity rating does {id} ({vendor} {product}) have?"]),
  "Q7_capec_attack": dict(harm="H1", answer_type="technique_set", phrasings=[
      "Which ATT&CK technique(s) does the attack pattern {id} ({name}) map to?",
      "What MITRE ATT&CK technique corresponds to {id} ({name})?",
      "{name} ({id}) is mapped to which ATT&CK technique(s)?"]),
}
print({k: v["harm"] for k, v in TEMPLATES.items()})

{'Q1_tech_tactic': 'H1', 'Q2_subtech_parent': 'H1', 'Q3_tech_mitigation': 'H2', 'Q4_group_uses': 'H5', 'Q5_cve_cwe': 'H4', 'Q6_cve_severity': 'H3', 'Q7_capec_attack': 'H1'}


## 8. Sinh câu hỏi

Mỗi câu hỏi là một bản ghi đầy đủ:

```text
qid, template, harm, answer_type, anchor (thực thể chính), question, question_variants,
gold (ID chuẩn), gold_display, distractor, facts (bộ ba KG chứng minh đáp án),
evidence_chunks, closed_world, split
```

**Lọc để chấm điểm sạch:**
- Q3 chỉ giữ kỹ thuật có 1–4 mitigation; danh sách dài quá thì F1 bị nhiễu và câu hỏi không thực tế.
- Q5 chỉ giữ CVE có 1–2 CWE.
- Q6 chỉ giữ CVE có CVSS v3.
- Q2 **không đưa ID vào câu hỏi**, vì ID của sub-technique đã lộ đáp án (T1001.001 hiển nhiên thuộc T1001). Chỉ giữ sub-technique có **tên duy nhất** trong ATT&CK; ví dụ tên "Local Account" xuất hiện dưới nhiều kỹ thuật cha nên bị loại.

In [11]:
questions = []
# Mỗi mẫu dùng một bộ sinh ngẫu nhiên RIÊNG (seed theo tên mẫu): sửa một mẫu không làm xáo các mẫu khác.
def rng_for(tpl):
    return random.Random(f"{SEED}:{tpl}")

def add_q(template, anchor, fmt, gold, distractor, facts, gold_display, closed_world=False, evidence=None, aux=None):
    T = TEMPLATES[template]
    variants = [p.format(**fmt) for p in T["phrasings"]]
    ev = set(evidence or [])
    for f in facts:
        ev |= fact2chunks.get(f, set())
    questions.append(dict(template=template, harm=T["harm"], answer_type=T["answer_type"], anchor=anchor,
                          question=variants[rng.randrange(len(variants))], question_variants=variants,
                          gold=sorted(gold), gold_display=gold_display, distractor=distractor,
                          facts=[list(f) for f in facts], evidence_chunks=sorted(ev), closed_world=closed_world,
                          aux=aux or {}))

# Q1: technique -> tactic(s)
rng = rng_for("Q1")
for t in techniques.values():
    if not t["tactics"]:
        continue
    tgt = rng.choice([a for a in ALL_TACTICS if a not in t["tactics"]])
    add_q("Q1_tech_tactic", t["id"], dict(name=t["name"], id=t["id"]), t["tactics"], tgt,
          [(t["id"], "in_tactic", a) for a in t["tactics"]], ", ".join(f"{TACTIC_NAME[a]} ({a})" for a in t["tactics"]))

rng = rng_for("Q2")
# Q2: sub-technique -> parent. Câu hỏi KHÔNG chứa ID (T1001.001 đã lộ cha là T1001),
# nên chỉ giữ các sub-technique có tên duy nhất trong ATT&CK để câu hỏi không mơ hồ.
name_count = collections.Counter(t["name"].lower() for t in techniques.values())
for t in techniques.values():
    if t["is_sub"] and t["parent"] and name_count[t["name"].lower()] == 1:
        tgt = same_tactic_distractor([t["parent"]], pool=parents)
        add_q("Q2_subtech_parent", t["id"], dict(name=t["name"], id=t["id"]), [t["parent"]], tgt,
              [(t["id"], "subtechnique_of", t["parent"])], tname(t["parent"]))

# Q3: technique -> mitigations (1–4)
rng = rng_for("Q3")
for t in techniques.values():
    ms = sorted(tech_mits.get(t["id"], ()))
    if 1 <= len(ms) <= 4:
        tgt = rng.choice([m for m in ALL_MITS if m not in ms])
        add_q("Q3_tech_mitigation", t["id"], dict(name=t["name"], id=t["id"]), ms, tgt,
              [(m, "mitigates", t["id"]) for m in ms], "; ".join(f"{m} {mitigations[m]['name']}" for m in ms))

rng = rng_for("Q4")
# Q4: group uses technique? (yes / hard-negative no, cân bằng theo nhóm)
for g in groups.values():
    used = sorted(uses_tech.get(g["id"], {}))
    if len(used) < 3:
        continue
    k = min(3, len(used))
    for t in rng.sample(used, k):
        add_q("Q4_group_uses", g["id"], dict(gname=g["name"], gid=g["id"], tname=techniques[t]["name"], tid=t),
              ["yes"], "no", [(g["id"], "uses", t)], "yes", aux={"technique": t})
    negs = set()
    for t in used:
        for a in techniques[t]["tactics"]:
            negs |= tech_by_tactic[a]
    negs = sorted(negs - set(used))
    for t in rng.sample(negs, min(k, len(negs))):
        # Câu "không": bằng chứng là toàn bộ các đoạn của tài liệu nhóm (danh sách đầy đủ kỹ thuật đã ghi nhận)
        add_q("Q4_group_uses", g["id"], dict(gname=g["name"], gid=g["id"], tname=techniques[t]["name"], tid=t),
              ["no"], "yes", [], "no", closed_world=True, evidence=doc2chunks[g["id"]], aux={"technique": t})

rng = rng_for("Q5")
# Q5: CVE -> CWE (1–2). v1.1: chỉ giữ CVE mà MỌI CWE đều là Weakness hợp lệ trong danh mục CWE.
# Các mã cũ kiểu CWE-189 "Numeric Errors", CWE-264, CWE-399 là *category* (nhóm), MITRE khuyến cáo
# không dùng để ánh xạ lỗ hổng -> làm đáp án chuẩn thì không đáng tin.
q5_dropped = 0
for v in cves.values():
    if 1 <= len(v["cwes"]) <= 2 and CWE_CATALOG_OK and not all(cwe[c]["is_weakness"] for c in v["cwes"]):
        q5_dropped += 1
        continue
    if 1 <= len(v["cwes"]) <= 2:
        tgt = rng.choice([c for c in KEV_CWE_POOL if c not in v["cwes"]])
        add_q("Q5_cve_cwe", v["id"], dict(id=v["id"], vendor=v["vendor"], product=v["product"]), v["cwes"], tgt,
              [(v["id"], "has_weakness", c) for c in v["cwes"]], "; ".join(f"{c} {cwe[c]['name']}".strip() for c in v["cwes"]))

# Q6: CVE -> CVSS v3 severity
rng = rng_for("Q6")
for v in cves.values():
    if v["cvss"]:
        sev = v["cvss"]["severity"]
        add_q("Q6_cve_severity", v["id"], dict(id=v["id"], vendor=v["vendor"], product=v["product"]), [sev],
              SEV_TARGET[sev], [(v["id"], "cvss_severity", sev)], f"{sev} ({v['cvss']['score']:.1f}, nguồn {v['cvss']['source']})")

# Q7: CAPEC -> ATT&CK technique(s)
rng = rng_for("Q7")
for p in capec.values():
    if 1 <= len(p["techniques"]) <= 3:
        tgt = same_tactic_distractor(p["techniques"])
        if tgt:
            add_q("Q7_capec_attack", p["id"], dict(id=p["id"], name=p["name"]), p["techniques"], tgt,
                  [(p["id"], "maps_to", t) for t in p["techniques"]], "; ".join(tname(t) for t in p["techniques"]))

print("Ứng viên trước khi lấy mẫu:", collections.Counter(q["template"] for q in questions))
print("Q5 bị loại vì CWE là category/lỗi thời:", q5_dropped)

Ứng viên trước khi lấy mẫu: Counter({'Q6_cve_severity': 1667, 'Q5_cve_cwe': 1494, 'Q4_group_uses': 534, 'Q1_tech_tactic': 428, 'Q3_tech_mitigation': 331, 'Q2_subtech_parent': 264, 'Q7_capec_attack': 158})
Q5 bị loại vì CWE là category/lỗi thời: 63


## 9. Lấy mẫu và chia tập dev/test theo thực thể

- **Trần mỗi mẫu** là `MAX_PER_TEMPLATE`, để không mẫu nào lấn át các mẫu khác. Q4 được lấy theo **nhóm** (giữ cân bằng có/không).
- **Chia theo thực thể:** thực thể chính (anchor) được băm (SHA-1 + seed) để quyết định thuộc dev hay test. Mọi câu hỏi về cùng một kỹ thuật, CVE hay nhóm đều nằm cùng một tập. Nhờ vậy, khi em chỉnh siêu tham số của phòng thủ trên dev, kết quả trên test không bị rò rỉ thông tin.
- **Quy tắc vàng:** mọi điều chỉnh (ngưỡng phòng thủ, prompt, top-k) chỉ làm trên **dev**. Tập **test** chỉ chạy một lần cho kết quả cuối cùng trong bài báo.

In [12]:
def split_of(anchor):
    h = int(hashlib.sha1(f"{SEED}:{anchor}".encode()).hexdigest(), 16) % 100
    return "dev" if h < DEV_PERCENT else "test"

final = []
for tpl in TEMPLATES:
    srng = random.Random(f"{SEED}:sample:{tpl}")   # bộ sinh riêng cho từng mẫu
    qs = [q for q in questions if q["template"] == tpl]
    if tpl == "Q4_group_uses":
        by_g = collections.defaultdict(list)
        for q in qs:
            by_g[q["anchor"]].append(q)
        gs = sorted(by_g); srng.shuffle(gs)
        picked = []
        for g in gs:
            if len(picked) + len(by_g[g]) > MAX_PER_TEMPLATE:
                break
            picked += by_g[g]
    else:
        qs = sorted(qs, key=lambda q: q["anchor"]); srng.shuffle(qs)
        picked = qs[:MAX_PER_TEMPLATE]
    final += picked

for i, q in enumerate(sorted(final, key=lambda q: (q["template"], q["anchor"], q["question"]))):
    q["qid"] = f"{q['template'].split('_')[0]}-{i:05d}"
    q["split"] = split_of(q["anchor"])
final = sorted(final, key=lambda q: q["qid"])

qdf = pd.DataFrame(final)
summary = qdf.pivot_table(index=["harm", "template"], columns="split", values="qid", aggfunc="count", fill_value=0)
summary["total"] = summary.sum(axis=1)
summary.loc[("", "TỔNG"), :] = summary.sum()
summary.astype(int)

split                    dev  test  total
harm template                            
H1   Q1_tech_tactic       76   324    400
     Q2_subtech_parent    43   221    264
     Q7_capec_attack      31   127    158
H2   Q3_tech_mitigation   57   274    331
H3   Q6_cve_severity      79   321    400
H4   Q5_cve_cwe           80   320    400
H5   Q4_group_uses        60   336    396
     TỔNG                426  1923   2349

## 10. Module chấm điểm (dùng chung cho các notebook sau)

Module được ghi ra file `ctibench_scoring.py` để các notebook sau `import` lại, bảo đảm **một định nghĩa chỉ số duy nhất** cho toàn bộ bài báo.

| Chỉ số | Định nghĩa |
|---|---|
| `em` | Tập ID trích từ câu trả lời **bằng đúng** tập gold |
| `f1` | F1 giữa tập trích ra và tập gold (cho câu nhiều đáp án) |
| `gold_hit` | Câu trả lời có nêu **ít nhất một** đáp án đúng |
| `distractor_hit` | Câu trả lời có nêu **đáp án nhiễu** cùng kiểu |

Notebook 03–04 sẽ bổ sung chỉ số riêng cho đề tài: câu trả lời có dùng **mã đã lỗi thời** (đúng theo phiên bản cũ, sai theo phiên bản hiện hành) hay không.

Trích đáp án bằng biểu thức chính quy trên ID chuẩn (T1234.001, TA0005, M1040, CWE-79, …). Riêng tactic có so khớp thêm theo **tên** (15 tên, không trùng nhau), vì LLM hay trả lời bằng tên. Tên tactic lấy theo **đúng phiên bản đang chạy**: ở v19.2, "Defense Evasion" không được ánh xạ (là tên cũ); ở v18.1 thì đó là tên đúng của TA0005.

In [13]:
%%writefile ctibench_scoring.py
"""Chấm điểm benchmark theo phiên bản (notebook 02). Dùng chung cho các notebook sau."""
from __future__ import annotations
import re

ID_RE = {
    "technique_set":  re.compile(r"\bT\d{4}(?:\.\d{3})?\b"),
    "tactic_set":     re.compile(r"\bTA\d{4}\b"),
    "mitigation_set": re.compile(r"\bM\d{4}\b"),
    "cwe_set":        re.compile(r"\bCWE-\d+\b", re.I),
}
SEVERITIES = ["CRITICAL", "HIGH", "MEDIUM", "LOW"]

def extract(answer: str, answer_type: str, tactic_names: dict | None = None) -> set:
    """Trích tập đáp án chuẩn hoá từ câu trả lời tự do của LLM."""
    a = answer or ""
    if answer_type == "yes_no":
        m = re.search(r"\b(yes|no)\b", a, re.I)
        return {m.group(1).lower()} if m else set()
    if answer_type == "severity":
        hits = [(m.start(), m.group(0).upper()) for m in re.finditer(r"\b(critical|high|medium|low)\b", a, re.I)]
        return {min(hits)[1]} if hits else set()          # lấy mức xuất hiện đầu tiên
    out = {x.upper() for x in ID_RE[answer_type].findall(a)}
    if answer_type == "tactic_set" and tactic_names:
        low = a.lower()
        for tid, name in tactic_names.items():
            if re.search(r"\b" + re.escape(name.lower()) + r"\b", low):
                out.add(tid)
    return out

def score(q: dict, answer: str, tactic_names: dict | None = None) -> dict:
    pred = extract(answer, q["answer_type"], tactic_names)
    gold, distractor = set(q["gold"]), q["distractor"]
    tp = len(pred & gold)
    p = tp / len(pred) if pred else 0.0
    r = tp / len(gold) if gold else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    gold_hit, distractor_hit = tp > 0, distractor in pred
    return dict(pred=sorted(pred), em=pred == gold, precision=p, recall=r, f1=f1,
                gold_hit=gold_hit, distractor_hit=distractor_hit)

def cohen_kappa(a: list, b: list) -> float:
    """Độ đồng thuận giữa hai người chấm (nhãn rời rạc)."""
    n = len(a); labels = set(a) | set(b)
    po = sum(x == y for x, y in zip(a, b)) / n
    pe = sum((a.count(l) / n) * (b.count(l) / n) for l in labels)
    return (po - pe) / (1 - pe) if pe < 1 else 1.0

Overwriting ctibench_scoring.py


In [14]:
import importlib, ctibench_scoring
importlib.reload(ctibench_scoring)
from ctibench_scoring import score, extract, cohen_kappa

# Kiểm thử đơn vị: chạy lại mỗi khi sửa module
q_tac = dict(answer_type="tactic_set", gold=["TA0004", "TA0005"], distractor="TA0002")
ta5 = TACTIC_NAME["TA0005"]          # "Defense Evasion" ở v18 trở về trước, "Stealth" từ v19
assert score(q_tac, f"It belongs to {ta5} (TA0005) and Privilege Escalation.", TACTIC_NAME)["em"]
if ta5 == "Stealth":
    assert not score(q_tac, "Defense Evasion", TACTIC_NAME)["gold_hit"]   # từ v19: tên cũ là SAI
else:
    assert score(q_tac, "Defense Evasion", TACTIC_NAME)["gold_hit"]       # trước v19: tên này là ĐÚNG
assert score(q_tac, "Execution (TA0002)", TACTIC_NAME)["distractor_hit"]
q_t = dict(answer_type="technique_set", gold=["T1055"], distractor="T1574")
assert extract("See T1055.011 and T1055", "technique_set") == {"T1055.011", "T1055"}
assert score(q_t, "The parent is T1055 Process Injection.")["em"]
assert score(q_t, "T1574, not T1055")["distractor_hit"] and score(q_t, "T1574, not T1055")["gold_hit"]
q_sev = dict(answer_type="severity", gold=["CRITICAL"], distractor="LOW")
assert score(q_sev, "Critical, base score 9.8")["em"] and score(q_sev, "It is rated low.")["distractor_hit"]
q_yn = dict(answer_type="yes_no", gold=["no"], distractor="yes")
assert score(q_yn, "No, ATT&CK does not list it.")["em"]
assert score(dict(answer_type="cwe_set", gold=["CWE-502"], distractor="CWE-79"), "cwe-502")["em"]
assert abs(cohen_kappa([1, 1, 0, 0], [1, 1, 0, 1]) - 0.5) < 1e-9
print("Tất cả kiểm thử chấm điểm đều qua ✓")

Tất cả kiểm thử chấm điểm đều qua ✓


## 11. Kiểm định chất lượng benchmark

Phản biện sẽ hỏi: *"Làm sao biết benchmark của bạn đúng?"* Notebook trả lời bằng các kiểm tra tự động sau. **Cell này phải chạy qua hết thì mới được dùng benchmark.**

1. **Truy vết đáp án về KG:** truy vấn lại KG cho từng câu và so với gold.
2. **Có bằng chứng:** câu nào cũng có ít nhất một đoạn chứa sự kiện, nên về nguyên tắc truy hồi được.
3. **Distractor hợp lệ:** cùng kiểu và không nằm trong gold.
4. **Không rò rỉ:** không thực thể nào xuất hiện ở cả dev lẫn test.
5. **Không trùng lặp:** không có hai câu hỏi giống hệt nhau.
6. **Bằng chứng thật sự chứa đáp án:** ID đáp án (hoặc mức nghiêm trọng) xuất hiện trong văn bản của ít nhất một đoạn bằng chứng.

Ngoài kiểm tra tự động còn **kiểm định thủ công**: xuất `HUMAN_AUDIT_N` câu ngẫu nhiên. Em và một đồng nghiệp chấm độc lập xem câu hỏi có rõ và đáp án có đúng không, rồi tính Cohen's κ. Số liệu này đưa vào bài, ví dụ: *"200 câu được hai chuyên gia kiểm tra, độ chính xác 99,0%, κ = 0,87"*.

In [15]:
chunk_text = {c["_id"]: c["text"] for c in corpus}
problems = collections.defaultdict(list)

def kg_gold(q):
    a, t = q["anchor"], q["template"]
    if t == "Q1_tech_tactic":     return kg_targets(a, "in_tactic")
    if t == "Q2_subtech_parent":  return kg_targets(a, "subtechnique_of")
    if t == "Q3_tech_mitigation": return sorted(s for s, _, r in KG.in_edges(a, data="rel") if r == "mitigates")
    if t == "Q5_cve_cwe":         return kg_targets(a, "has_weakness")
    if t == "Q6_cve_severity":    return [KG.nodes[a]["cvss_severity"]]
    if t == "Q7_capec_attack":    return kg_targets(a, "maps_to")
    if t == "Q4_group_uses":
        return ["yes"] if q["aux"]["technique"] in kg_targets(a, "uses") else ["no"]

for q in final:
    if kg_gold(q) != q["gold"]:                       problems["1_kg_mismatch"].append(q["qid"])
    if not q["evidence_chunks"]:                      problems["2_no_evidence"].append(q["qid"])
    if q["distractor"] in q["gold"] or q["distractor"] is None: problems["3_bad_distractor"].append(q["qid"])
    if not q["closed_world"]:
        ev = " ".join(chunk_text[c] for c in q["evidence_chunks"])
        must = [q["aux"]["technique"]] if q["answer_type"] == "yes_no" else q["gold"]
        if not all(g in ev for g in must):            problems["6_answer_not_in_evidence"].append(q["qid"])

anchors = pd.DataFrame(final).groupby("anchor")["split"].nunique()
problems["4_leakage"] = list(anchors[anchors > 1].index)
dup = pd.DataFrame(final).question.duplicated()
problems["5_duplicates"] = list(pd.DataFrame(final)[dup].qid)

for k in ["1_kg_mismatch", "2_no_evidence", "3_bad_distractor", "4_leakage", "5_duplicates", "6_answer_not_in_evidence"]:
    print(f"{k:28s}: {len(problems[k])} lỗi", problems[k][:5])
assert not any(problems.values()), "Benchmark CHƯA đạt kiểm định — sửa trước khi dùng"
print("\nBenchmark đạt toàn bộ kiểm định tự động ✓")

1_kg_mismatch               : 0 lỗi []
2_no_evidence               : 0 lỗi []
3_bad_distractor            : 0 lỗi []
4_leakage                   : 0 lỗi []
5_duplicates                : 0 lỗi []
6_answer_not_in_evidence    : 0 lỗi []

Benchmark đạt toàn bộ kiểm định tự động ✓


In [16]:
# Thống kê mô tả (đưa vào bảng Dataset của bài báo)
fdf = pd.DataFrame(final)
fdf["n_gold"] = fdf.gold.str.len()
fdf["n_evidence"] = fdf.evidence_chunks.str.len()
desc = fdf.groupby("template").agg(n=("qid", "count"), gold_mean=("n_gold", "mean"),
                                   evidence_median=("n_evidence", "median"), evidence_max=("n_evidence", "max"))
print("Cân bằng có/không ở Q4:", collections.Counter(fdf[fdf.template == "Q4_group_uses"].gold.str[0]))
print("Phân bố mức nghiêm trọng ở Q6:", collections.Counter(fdf[fdf.template == "Q6_cve_severity"].gold.str[0]))
# Baseline "đoán đáp án phổ biến nhất": mọi kết quả LLM phải vượt mức này mới có ý nghĩa.
# KEV thiên về lỗ hổng nặng, nên ở Q6 đoán "HIGH" đã đúng khoảng một nửa — phải báo cáo con số này trong bài.
desc["majority_baseline_em"] = fdf.groupby("template").gold.apply(lambda g: g.map(tuple).value_counts(normalize=True).iloc[0])
desc.round(2)

Cân bằng có/không ở Q4: Counter({'yes': 198, 'no': 198})
Phân bố mức nghiêm trọng ở Q6: Counter({'HIGH': 225, 'CRITICAL': 125, 'MEDIUM': 46, 'LOW': 4})


,n,gold_mean,evidence_median,evidence_max,majority_baseline_em
template,,,,,
Q1_tech_tactic,400,1.35,1.0,1,0.16
Q2_subtech_parent,264,1.00,1.0,1,0.06
Q3_tech_mitigation,331,2.10,3.0,5,0.06
Q4_group_uses,396,1.00,1.0,6,0.50
Q5_cve_cwe,400,1.08,1.0,1,0.08
Q6_cve_severity,400,1.00,1.0,1,0.56
Q7_capec_attack,158,1.28,1.0,1,0.06


## 12. Xuất benchmark

```text
benchmark/attack-<phiên bản>/
├── questions.jsonl          ← bản ghi đầy đủ (dùng cho notebook 03–05)
├── corpus.jsonl             ← kho RAG sạch, định dạng BEIR (_id, title, text)
├── queries.jsonl            ← định dạng BEIR (_id, text)
├── qrels/dev.tsv, test.tsv  ← định dạng BEIR (query-id, corpus-id, score)
├── human_audit.csv          ← mẫu kiểm định thủ công
├── manifest.json            ← phiên bản, SHA-256, số lượng, seed
└── DATASHEET.md             ← mô tả bộ dữ liệu theo khung Datasheets for Datasets
data/processed/attack-<phiên bản>/kg/  ← đồ thị tri thức của phiên bản
```

In [17]:
def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

write_jsonl(BENCH / "questions.jsonl", final)
write_jsonl(BENCH / "corpus.jsonl", [{"_id": c["_id"], "title": c["title"], "text": c["text"],
                                      "metadata": {"doc_id": c["doc_id"], "kind": c["kind"]}} for c in corpus])
write_jsonl(BENCH / "queries.jsonl", [{"_id": q["qid"], "text": q["question"],
                                       "metadata": {"template": q["template"], "split": q["split"]}} for q in final])
for sp in ("dev", "test"):
    rows = [(q["qid"], c, 1) for q in final if q["split"] == sp for c in q["evidence_chunks"]]
    pd.DataFrame(rows, columns=["query-id", "corpus-id", "score"]).to_csv(BENCH / "qrels" / f"{sp}.tsv", sep="\t", index=False)

# Mẫu kiểm định thủ công: phân tầng theo template
arng = random.Random(SEED + 2)
per = max(1, HUMAN_AUDIT_N // len(TEMPLATES))
audit = []
for tpl in TEMPLATES:
    pool = [q for q in final if q["template"] == tpl]
    for q in arng.sample(pool, min(per, len(pool))):
        audit.append(dict(qid=q["qid"], template=tpl, question=q["question"], gold=q["gold_display"],
                          evidence_excerpt=chunk_text[q["evidence_chunks"][0]][:600],
                          annotator_A_correct="", annotator_B_correct="", notes=""))
pd.DataFrame(audit).to_csv(BENCH / "human_audit.csv", index=False)

MANIFEST["counts"] = {"questions": len(final), "by_template": dict(collections.Counter(q["template"] for q in final)),
                      "by_split": dict(collections.Counter(q["split"] for q in final)),
                      "corpus_chunks": len(corpus), "corpus_docs": len({c['doc_id'] for c in corpus}),
                      "kg_nodes": KG.number_of_nodes(), "kg_edges": KG.number_of_edges()}
MANIFEST["config"] = dict(MAX_KEV_CVES=MAX_KEV_CVES, MAX_PER_TEMPLATE=MAX_PER_TEMPLATE, DEV_PERCENT=DEV_PERCENT,
                          CHUNK_MAX_WORDS=CHUNK_MAX_WORDS)
MANIFEST["output_sha256"] = {p.name: sha256_file(p) for p in [BENCH / "questions.jsonl", BENCH / "corpus.jsonl"]}
(BENCH / "manifest.json").write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False))
print(json.dumps(MANIFEST["counts"], indent=2, ensure_ascii=False))

{
  "questions": 2349,
  "by_template": {
    "Q1_tech_tactic": 400,
    "Q2_subtech_parent": 264,
    "Q3_tech_mitigation": 331,
    "Q4_group_uses": 396,
    "Q5_cve_cwe": 400,
    "Q6_cve_severity": 400,
    "Q7_capec_attack": 158
  },
  "by_split": {
    "dev": 426,
    "test": 1923
  },
  "corpus_chunks": 4017,
  "corpus_docs": 3815,
  "kg_nodes": 4247,
  "kg_edges": 7020
}


In [18]:
src = MANIFEST["sources"]
datasheet = f'''# DATASHEET — CTI-RAG Benchmark (bản {datetime.date.today().isoformat()})

## Motivation
Evaluate knowledge-poisoning robustness of retrieval-augmented CTI assistants with verifiable, structured ground truth.

## Composition
- {len(final)} questions over {len(TEMPLATES)} templates and 5 harm categories (H1–H5); splits: {dict(collections.Counter(q["split"] for q in final))}.
- Clean corpus: {len(corpus)} chunks from {len({c["doc_id"] for c in corpus})} entity documents (ATT&CK techniques, mitigations, groups; KEV CVEs; CWE; CAPEC).
- Each question: gold answer (canonical IDs), type-consistent distractor answer, supporting KG facts, evidence chunk IDs.

## Collection process
- MITRE ATT&CK Enterprise v{ATTACK_VERSION} (STIX 2.1), sha256 {src["attack"]["sha256"][:16]}…
- CAPEC STIX 2.1 (mitre/cti), sha256 {src["capec"]["sha256"][:16]}…
- CISA KEV catalog version {src["kev"]["catalog_version"]} ({src["kev"]["count"]} entries)
- CVE List V5 records for KEV CVEs ({src["cvelistV5"]["n_records"]} fetched); CVSS v3 from CNA, else CISA-ADP (Vulnrichment)
- CWE catalog: {"cwec_latest.xml (" + str(src["cwe"].get("n_weaknesses", "")) + " weaknesses)" if "cwe" in src else "not available; CWE names taken from CVE problemTypes"}

## Preprocessing
Revoked/deprecated objects removed; citations and markdown links stripped; documents rendered per entity and chunked
(≤{CHUNK_MAX_WORDS} words, header repeated per chunk). Questions generated from templates; answers derived deterministically from the KG.
Group-use questions are closed-world relative to ATT&CK ("documented as using").

## Uses / limitations
Template questions are simpler and more uniform than real analyst queries; complement with a small expert-written set.
LLM parametric knowledge may answer some questions without retrieval — always report a closed-book baseline.
KEV-only CVE subset; not representative of all CVEs.

## Distribution & licensing
Derived from MITRE ATT&CK, CAPEC, CWE (MITRE terms of use, attribution required), CISA KEV (U.S. government work), CVE List (CVE terms of use).
Verify each source's current terms before redistribution.

## Maintenance
Regenerate with notebook 02 using the pinned versions in manifest.json; any version change produces a new benchmark release.
'''
(BENCH / "DATASHEET.md").write_text(datasheet, encoding="utf-8")
print("Đã ghi", (BENCH / "DATASHEET.md").relative_to(ROOT))

Đã ghi benchmark/attack-7.2/DATASHEET.md


In [19]:
# Xem vài câu hỏi mẫu của từng template
show = fdf.groupby("template").head(1)[["qid", "harm", "question", "gold_display", "distractor", "n_evidence", "split"]]
show

,qid,harm,question,gold_display,distractor,n_evidence,split
0,Q1-00000,H1,Under which ATT&CK Enterprise tactic(s) is T1001 (Data Obfuscation) categorized?,Command and Control (TA0011),TA0040,1,dev
400,Q2-00400,H1,"Which ATT&CK technique has ""Junk Data"" as one of its sub-techniques?",T1001 Data Obfuscation,T1573,1,test
664,Q3-00664,H2,"To defend against Data Obfuscation (T1001), which ATT&CK mitigations apply?",M1031 Network Intrusion Prevention,M1025,2,dev
995,Q4-00995,H5,"According to MITRE ATT&CK, is the group Axiom (G0001) documented as using Accessibilit...",yes,no,1,test
1391,Q5-01391,H4,CVE-2006-2492 affecting Microsoft Word maps to which CWE?,CWE-120 Buffer Copy without Checking Size of Input ('Classic Buffer Overflow'),CWE-330,1,test
1791,Q6-01791,H3,How severe is CVE-2002-0367 in Microsoft Windows according to its CVSS v3 base score: ...,"HIGH (7.8, nguồn CISA-ADP)",LOW,1,test
2191,Q7-02191,H1,Which ATT&CK technique(s) does the attack pattern CAPEC-1 (Accessing Functionality Not...,T1574.010 Services File Permissions Weakness,T1505.002,1,test


## 13. Tự đánh giá theo tiêu chuẩn và việc tiếp theo

| Tiêu chí phản biện thường dùng | Notebook 02 đáp ứng thế nào | Còn phải làm |
|---|---|---|
| Đáp án đúng có kiểm chứng được không? | Sinh tất định từ KG; kiểm tra tự động 6 điểm | Kiểm định thủ công 200 câu, báo cáo độ chính xác và κ |
| Tái lập được không? | Phiên bản ghim, SHA-256, seed, manifest | Công bố code và benchmark (GitHub + Zenodo DOI) khi nộp bài |
| Có rò rỉ dev/test không? | Chia theo thực thể, đã kiểm tra | — |
| Câu hỏi có quá dễ hay đơn điệu không? | 3 cách diễn đạt mỗi mẫu; distractor là hard negative | Thêm 50–100 câu do người viết, phong cách nhà phân tích thực tế; báo cáo kết quả riêng |
| LLM có tự biết đáp án không? | Đã nêu rủi ro (ví dụ đổi tên tactic ở v19) | **Notebook 04:** baseline không dùng RAG |
| Có so được với công trình trước không? | Định dạng BEIR chuẩn | So với VersionRAG và các baseline ở notebook 05 |
| Tài liệu hoá bộ dữ liệu? | `DATASHEET.md` | Bổ sung số liệu kiểm định thủ công |

### Việc của em sau khi chạy notebook này

1. Chạy notebook với `ATTACK_VERSION = "19.2"`, sau đó đổi thành `"18.1"` và chạy lại. Kiểm tra mục 11 đạt ở **cả hai** lần.
2. Ghi mã SHA-256 trong `benchmark/attack-<phiên bản>/manifest.json` của cả hai lần vào sổ thí nghiệm.
3. Gửi thầy hai notebook đã chạy (lưu bản 18.1 với tên khác, ví dụ `02_..._v18.1.ipynb`).

**Notebook 03 (tiếp theo):** ghép câu hỏi của hai phiên bản theo danh sách thay đổi từ notebook 01, tạo bộ câu hỏi "trước/sau" có nhãn kiểu thay đổi cho pilot Trụ 2.